# Migrating from GPT-4o to a GPT-5 reasoning model

GPT-4o is a non-reasoning chat model. The GPT-5 family flagships (here, `gpt-5-mini`) are
**reasoning** models: they spend hidden internal tokens *thinking* before they answer. That
one difference, plus the move from Chat Completions to the Responses API, is the source of
almost every migration surprise.

This notebook migrates a GPT-4o workload to `gpt-5-mini` **live, through the same APIM
gateway**, and reproduces each gotcha with a real call so you can see the exact error or
behaviour rather than read about it.

| | GPT-4o | GPT-5 family (reasoning) |
|---|---|---|
| Model class | Non-reasoning chat | Reasoning (thinks first) |
| Hidden reasoning tokens | none | yes, billed as output |
| Sampling controls (`temperature`, `top_p`, penalties) | supported | rejected (defaults only) |
| Output cap parameter | `max_tokens` | `max_completion_tokens` / `max_output_tokens` |
| Recommended API | Chat Completions | Responses |
| Context window | 128K total / 16K out | 272K in / 128K out |

> Both models are reached through the core APIM gateway. `gpt-5-mini` is already deployed on
> the core account (by [08-10b](../08-agents/08-10b-hosted-copilot-sdk-agent-multi/08-10b-00-hosted-copilot-sdk-agent-multi.md));
> Step 1 deploys `gpt-4o` alongside it. Deeper background:
> [`docs/gpt-4o-to-gpt-5.1-migration-research.md`](../docs/gpt-4o-to-gpt-5.1-migration-research.md).

## Prerequisites

1. **`05-02` core gateway deployed** so the repo `.env` has `GATEWAY_URL` and `ALPHA_GATEWAY_KEY`.
2. **`az login`** with rights to create a model deployment on `rg-foundry-core-{suffix}`
   (Cognitive Services Contributor) - Step 1 deploys `gpt-4o` on the core account.
3. **`gpt-5-mini` already deployed** on the core account `aif-core-{suffix}` - it is the
   reasoning model this lab migrates to (deployed by `08-10b`).
4. **Quota** for `gpt-4o` GlobalStandard in the gateway region (`eastus2`).
5. The repo `uv` environment (`openai>=2`, `azure-identity`, `python-dotenv`).

## Configuration

Names are derived from the subscription-based suffix (matching `05-02`); the gateway endpoint
and key come from `.env`. A single `AzureOpenAI` client pointed at the gateway will serve both
models and both API surfaces.

In [1]:
import os, subprocess, hashlib, time
from pathlib import Path
from dotenv import load_dotenv

repo_root = Path(subprocess.run(
    "git rev-parse --show-toplevel", shell=True, capture_output=True, text=True
).stdout.strip())
load_dotenv(repo_root / ".env", override=True)

SUBSCRIPTION_ID = subprocess.run(
    "az account show --query id -o tsv", shell=True, capture_output=True, text=True
).stdout.strip()
SUFFIX   = hashlib.sha256((SUBSCRIPTION_ID + "v2").encode()).hexdigest()[:6]
LOCATION = "eastus2"

CORE_ACCOUNT = f"aif-core-{SUFFIX}"        # core/admin Foundry account behind the gateway
CORE_RG      = f"rg-foundry-core-{SUFFIX}"

# APIM gateway (set by 05-02)
GATEWAY_URL      = os.environ["GATEWAY_URL"]            # https://apim-foundry-{suffix}.azure-api.net/openai
GATEWAY_KEY      = os.environ["ALPHA_GATEWAY_KEY"]      # team APIM subscription key
GATEWAY_ENDPOINT = GATEWAY_URL.removesuffix("/openai")  # base URL for the AzureOpenAI client

# The legacy chat model (deployed in Step 1) and the reasoning model (already on core)
GPT4O_MODEL   = os.environ.get("GPT4O_MODEL", "gpt-4o")
GPT4O_VERSION = "2024-11-20"
GPT5_MODEL    = os.environ.get("GPT5_MODEL", "gpt-5-mini")

# A preview api-version is required for reasoning params and the Responses API through the gateway
API_VERSION = "2025-04-01-preview"

print(f"Suffix           : {SUFFIX}")
print(f"Core account     : {CORE_ACCOUNT}  (rg: {CORE_RG})")
print(f"Gateway endpoint : {GATEWAY_ENDPOINT}")
print(f"Legacy model     : {GPT4O_MODEL} ({GPT4O_VERSION})")
print(f"Reasoning model  : {GPT5_MODEL}")
print(f"API version      : {API_VERSION}")

Suffix           : c2676f
Core account     : aif-core-c2676f  (rg: rg-foundry-core-c2676f)
Gateway endpoint : https://apim-foundry-c2676f.azure-api.net
Legacy model     : gpt-4o (2024-11-20)
Reasoning model  : gpt-5-mini
API version      : 2025-04-01-preview


## Step 1 - Deploy the legacy GPT-4o model on the core account

A migration needs both models side by side. We deploy `gpt-4o` on the **core** account
`aif-core-{suffix}` - the account that backs the gateway's default route.

Because APIM's chat operation is a wildcard (`/deployments/{deployment-id}/chat/completions`)
and the `/responses` operation both fall through to the `openai` backend (the core account),
**a new deployment on the core account is reachable through the gateway immediately - no APIM
change is required.** Governance still holds: the model lives on the core account, not in a
spoke, so the `deny-model-deployments` policy is untouched.

The cell is idempotent: it skips deployment if `gpt-4o` already exists.

In [2]:
def _az(*args):
    return subprocess.run(["az", *args], capture_output=True, text=True)

deployed = _az("cognitiveservices", "account", "deployment", "list",
               "-n", CORE_ACCOUNT, "-g", CORE_RG, "--query", "[].name", "-o", "tsv").stdout.split()

if GPT4O_MODEL in deployed:
    print(f"'{GPT4O_MODEL}' already deployed on {CORE_ACCOUNT} - skipping.")
else:
    print(f"Deploying '{GPT4O_MODEL}' ({GPT4O_VERSION}) on {CORE_ACCOUNT} ...")
    r = _az("cognitiveservices", "account", "deployment", "create",
            "-n", CORE_ACCOUNT, "-g", CORE_RG,
            "--deployment-name", GPT4O_MODEL,
            "--model-name", "gpt-4o", "--model-version", GPT4O_VERSION, "--model-format", "OpenAI",
            "--sku-name", "GlobalStandard", "--sku-capacity", "30", "-o", "none")
    print("Deployed." if r.returncode == 0 else f"Failed:\n{r.stderr}")

names = _az("cognitiveservices", "account", "deployment", "list",
            "-n", CORE_ACCOUNT, "-g", CORE_RG, "--query", "[].name", "-o", "tsv").stdout.split()
print("\nCore account deployments:")
for n in names:
    tag = "  <- legacy (this lab)" if n == GPT4O_MODEL else ("  <- reasoning target" if n == GPT5_MODEL else "")
    print(f"  - {n}{tag}")

'gpt-4o' already deployed on aif-core-c2676f - skipping.

Core account deployments:
  - gpt-4.1-mini
  - text-embedding-3-large
  - gpt-4.1-mini-bank-guardrails
  - gpt-5-mini  <- reasoning target
  - gpt-4o  <- legacy (this lab)


## Step 2 - One gateway client for both models

A single `AzureOpenAI` client points at the APIM gateway with the team subscription key. APIM
authenticates to the core account with its managed identity, so the key never travels past the
gateway. The same client calls `gpt-4o` and `gpt-5-mini`, on both `chat.completions` and
`responses` - the only things that change during the migration are the model name, the
parameters, and the API surface.

In [3]:
from openai import AzureOpenAI

gateway = AzureOpenAI(
    azure_endpoint=GATEWAY_ENDPOINT,
    api_key=GATEWAY_KEY,
    api_version=API_VERSION,
    timeout=180,
)
print("gateway client ready ->", GATEWAY_ENDPOINT)

gateway client ready -> https://apim-foundry-c2676f.azure-api.net


## The baseline: GPT-4o today

A typical GPT-4o chat completion - a system prompt, a user message, `temperature`, and
`max_tokens`. This is the code we are migrating. Note the usage: there are **no reasoning
tokens**.

In [4]:
PROMPT = "What is catastrophic forgetting in neural networks? Answer in two sentences."

r = gateway.chat.completions.create(
    model=GPT4O_MODEL,
    messages=[
        {"role": "system", "content": "You are a concise technical assistant."},
        {"role": "user",   "content": PROMPT},
    ],
    temperature=0.7,
    max_tokens=200,
)
print(f"model             : {r.model}")
print(f"finish_reason     : {r.choices[0].finish_reason}")
print(f"completion_tokens : {r.usage.completion_tokens}")
print(f"reasoning_tokens  : {getattr(getattr(r.usage, 'completion_tokens_details', None), 'reasoning_tokens', 0)}")
print()
print(r.choices[0].message.content)

model             : gpt-4o-2024-11-20
finish_reason     : stop
completion_tokens : 49
reasoning_tokens  : 0

Catastrophic forgetting in neural networks refers to the phenomenon where a model forgets previously learned information upon learning new tasks, as the updates overwrite older knowledge. This occurs because standard neural networks lack mechanisms to retain and balance information across multiple tasks.


## The naive swap

The instinct is to change one string: `model="gpt-4o"` becomes `model="gpt-5-mini"`. The next
cells point the **same request** at the reasoning model and show, one at a time, what breaks
and why.

### Gotcha 1 - sampling parameters are rejected

Reasoning models do not accept the GPT-4o sampling controls. `temperature` is pinned to its
default (`1`); any other value, and `top_p` / `presence_penalty` / `frequency_penalty` /
`logprobs` / `logit_bias`, return HTTP 400. Steering moves to `reasoning_effort` and
`verbosity` instead.

In [5]:
# GPT-4o accepts custom sampling parameters
ok = gateway.chat.completions.create(
    model=GPT4O_MODEL,
    messages=[{"role": "user", "content": "Say hello in five words."}],
    temperature=0.2, top_p=0.5, frequency_penalty=0.5,
)
print(f"{GPT4O_MODEL}: temperature + top_p + frequency_penalty -> OK ({ok.choices[0].message.content.strip()})\n")

# The same parameters on the reasoning model are rejected
for param, kwargs in [
    ("temperature",       {"temperature": 0.2}),
    ("top_p",             {"top_p": 0.5}),
    ("frequency_penalty", {"frequency_penalty": 0.5}),
]:
    try:
        gateway.chat.completions.create(
            model=GPT5_MODEL,
            messages=[{"role": "user", "content": "Say hello in five words."}],
            max_completion_tokens=2000, **kwargs,
        )
        print(f"{GPT5_MODEL}: {param:17s} -> unexpectedly accepted")
    except Exception as e:
        print(f"{GPT5_MODEL}: {param:17s} -> {str(e)[:170]}")

gpt-4o: temperature + top_p + frequency_penalty -> OK (Hello there, how are you?)

gpt-5-mini: temperature       -> Error code: 400 - {'error': {'message': "Unsupported value: 'temperature' does not support 0.2 with this model. Only the default (1) value is supported.", 'type': 'invali
gpt-5-mini: top_p             -> Error code: 400 - {'error': {'message': "Unsupported parameter: 'top_p' is not supported with this model.", 'type': 'invalid_request_error', 'param': 'top_p', 'code': 'un
gpt-5-mini: frequency_penalty -> Error code: 400 - {'error': {'message': "Unsupported parameter: 'frequency_penalty' is not supported with this model.", 'type': 'invalid_request_error', 'param': 'frequen


### Gotcha 2 - `max_tokens` is renamed

Reasoning models reject `max_tokens`. The replacement is `max_completion_tokens` (Chat
Completions) or `max_output_tokens` (Responses). Watch for SDK wrappers that inject
`max_tokens` for you - they trigger this 400 even when you never set it.

In [6]:
try:
    gateway.chat.completions.create(
        model=GPT5_MODEL, messages=[{"role": "user", "content": PROMPT}], max_tokens=200,
    )
except Exception as e:
    print(f"max_tokens=200          -> {str(e)[:190]}")

r = gateway.chat.completions.create(
    model=GPT5_MODEL, messages=[{"role": "user", "content": PROMPT}], max_completion_tokens=2000,
)
print(f"max_completion_tokens   -> OK (finish={r.choices[0].finish_reason}, "
      f"completion_tokens={r.usage.completion_tokens})")

max_tokens=200          -> Error code: 400 - {'error': {'message': "Unsupported parameter: 'max_tokens' is not supported with this model. Use 'max_completion_tokens' instead.", 'type': 'invalid_request_error', 'param'
max_completion_tokens   -> OK (finish=stop, completion_tokens=200)


### Gotcha 3 - hidden reasoning tokens

The reasoning model generates internal tokens before its visible answer. They are **billed as
output tokens** and counted against rate limits, but never returned to you. They surface only
in `usage.completion_tokens_details.reasoning_tokens`. The same prompt that cost GPT-4o N
output tokens now costs N plus the hidden reasoning tokens.

In [7]:
def usage_line(label, r):
    d = getattr(r.usage, "completion_tokens_details", None)
    reasoning = getattr(d, "reasoning_tokens", 0) or 0
    visible = r.usage.completion_tokens - reasoning
    print(f"{label:11s} | completion_tokens={r.usage.completion_tokens:4d} | "
          f"reasoning={reasoning:4d} | visible={visible:4d}")

q = [{"role": "user", "content": "What is catastrophic forgetting? One sentence."}]
r4 = gateway.chat.completions.create(model=GPT4O_MODEL, messages=q, max_tokens=2000)
r5 = gateway.chat.completions.create(model=GPT5_MODEL, messages=q, max_completion_tokens=2000)
usage_line(GPT4O_MODEL, r4)
usage_line(GPT5_MODEL, r5)
print("\nThe reasoning tokens are billed but invisible - budget and cost for them.")

gpt-4o      | completion_tokens=  32 | reasoning=   0 | visible=  32
gpt-5-mini  | completion_tokens= 104 | reasoning=  64 | visible=  40

The reasoning tokens are billed but invisible - budget and cost for them.


### Gotcha 4 - the empty-answer trap

`max_completion_tokens` caps reasoning **plus** the visible answer. Set it too low and the
model can spend the entire budget thinking, returning empty content with
`finish_reason="length"` - and you still pay for the reasoning tokens. OpenAI suggests
reserving at least ~25,000 tokens while tuning. Below, a 16-token cap yields no answer at all.

In [8]:
r = gateway.chat.completions.create(
    model=GPT5_MODEL,
    messages=[{"role": "user", "content": "Explain backpropagation in detail."}],
    max_completion_tokens=16,
)
d = r.usage.completion_tokens_details
print("max_completion_tokens=16")
print(f"  finish_reason     : {r.choices[0].finish_reason}")
print(f"  content           : {r.choices[0].message.content!r}")
print(f"  completion_tokens : {r.usage.completion_tokens} "
      f"(reasoning={getattr(d, 'reasoning_tokens', 0)})  <- billed, no answer\n")

r = gateway.chat.completions.create(
    model=GPT5_MODEL,
    messages=[{"role": "user", "content": "Explain backpropagation in detail."}],
    max_completion_tokens=3000,
)
print("max_completion_tokens=3000")
print(f"  finish_reason     : {r.choices[0].finish_reason}")
print(f"  answer length     : {len(r.choices[0].message.content or '')} chars")

max_completion_tokens=16
  finish_reason     : length
  content           : ''
  completion_tokens : 16 (reasoning=16)  <- billed, no answer

max_completion_tokens=3000
  finish_reason     : stop
  answer length     : 5677 chars


### Gotcha 5 - reasoning effort drives cost and latency

`reasoning_effort` is the new throttle. Lower effort means fewer reasoning tokens, lower cost,
and lower latency; higher effort means deeper thinking on hard problems. Watch the reasoning
tokens and wall-clock time climb together across the levels.

In [9]:
prompt = ("A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. "
          "How much does the ball cost? Show your reasoning.")
print(f"{'effort':8s} | {'latency':>8s} | {'reasoning':>9s} | {'completion':>10s}")
print("-" * 46)
for effort in ["none", "minimal", "low", "medium", "high"]:
    try:
        t0 = time.perf_counter()
        r = gateway.chat.completions.create(
            model=GPT5_MODEL, messages=[{"role": "user", "content": prompt}],
            reasoning_effort=effort, max_completion_tokens=4000,
        )
        dt = time.perf_counter() - t0
        d = r.usage.completion_tokens_details
        print(f"{effort:8s} | {dt:7.1f}s | {getattr(d, 'reasoning_tokens', 0):9d} | "
              f"{r.usage.completion_tokens:10d}")
    except Exception as e:
        print(f"{effort:8s} | error: {str(e)[:80]}")

effort   |  latency | reasoning | completion
----------------------------------------------
none     |     2.0s |         0 |         96
minimal  |     2.1s |         0 |        118
low      |     2.8s |        64 |        180
medium   |     5.5s |       256 |        421
high     |     6.8s |       448 |        567


The supported effort values vary by model and version: `minimal` is the original GPT-5 floor,
while `none` (a true non-reasoning mode, like GPT-4o) is documented for GPT-5.1 and later. The
reliable approach is exactly what the cell above does - probe the deployment. For a
latency-sensitive GPT-4o workload, the natural target is `gpt-5.1` at `reasoning_effort="none"`
(or `gpt-5-mini` at `minimal`), then raise effort only on routes where quality needs it. See
[`docs/gpt-4o-to-gpt-5.1-migration-research.md`](../docs/gpt-4o-to-gpt-5.1-migration-research.md).

### Gotcha 6 - `system` vs `developer` role

Reasoning models use the `developer` role for instructions, but the latest models still accept
a `system` message and map it across, so existing GPT-4o code keeps working. Do not send both
roles in one request.

In [10]:
r = gateway.chat.completions.create(
    model=GPT5_MODEL,
    messages=[
        {"role": "system", "content": "You answer in exactly one word."},
        {"role": "user",   "content": "What is the capital of France?"},
    ],
    max_completion_tokens=2000,
)
print(f"system message accepted -> {r.choices[0].message.content!r}")

system message accepted -> 'Paris'


## The API surface shift: Chat Completions to Responses

Everything so far used Chat Completions, which still works with GPT-5. But OpenAI recommends
the **Responses API** for reasoning models: it is stateful, it can reuse reasoning across
turns (better quality and lower cost in multi-step/agentic flows), and it has a simpler
`input` field. The same gateway client exposes it - only the method changes.

Two differences to note immediately: you read `response.output_text` (not
`choices[0].message.content`), and reasoning effort is passed as `reasoning={"effort": ...}`.

In [11]:
r = gateway.responses.create(
    model=GPT5_MODEL,
    input="What is catastrophic forgetting? One sentence.",
    reasoning={"effort": "low"},
    max_output_tokens=2000,
)
d = getattr(r.usage, "output_tokens_details", None)
print(f"status      : {r.status}")
print(f"output_text : {r.output_text}")
print(f"usage       : input={r.usage.input_tokens}, output={r.usage.output_tokens}, "
      f"reasoning={getattr(d, 'reasoning_tokens', 0)}")

status      : completed
output_text : Catastrophic forgetting is the phenomenon where a machine learning model rapidly and dramatically loses performance on previously learned tasks after being trained on new tasks or data.
usage       : input=14, output=83, reasoning=0


### Multi-turn with `previous_response_id`

Pass `previous_response_id` to chain turns server-side. The model gets the prior context (and,
on reasoning models, the prior reasoning items) without you resending the history - the key
reason Responses outperforms Chat Completions on multi-step work.

In [12]:
first = gateway.responses.create(
    model=GPT5_MODEL,
    input="Define catastrophic forgetting in one sentence.",
    reasoning={"effort": "low"}, max_output_tokens=2000,
)
second = gateway.responses.create(
    model=GPT5_MODEL,
    previous_response_id=first.id,
    input=[{"role": "user", "content": "Now explain it to a 10-year-old."}],
    reasoning={"effort": "low"}, max_output_tokens=2000,
)
print("turn 1:", first.output_text)
print()
print("turn 2 (chained via previous_response_id):", second.output_text)

turn 1: Catastrophic forgetting is the sudden and severe loss of previously learned knowledge in a machine learning model when it is trained on new tasks or data, typically because the new training updates overwrite the parameters that encoded earlier information.

turn 2 (chained via previous_response_id): Imagine your brain is a box of crayons where each color is a skill you learned. Catastrophic forgetting is like when you learn to use a brand-new crayon color, but while putting it in the box you accidentally smudge or cover up a lot of the old colors so you suddenly can't use them anymore. For computers that learn, it means when they learn something new they sometimes accidentally "erase" what they learned before and can't remember it anymore.


### Verbosity controls the answer length

`verbosity` (low / medium / high) sets how long the **final answer** is, independent of how
much the model thinks. It is most effective on the Responses API via `text={"verbosity": ...}`.
Reasoning effort is held at `minimal` so only verbosity moves the output size.

In [13]:
print(f"{'verbosity':10s} | {'output_tokens':>13s} | {'chars':>6s}")
print("-" * 36)
for v in ["low", "medium", "high"]:
    r = gateway.responses.create(
        model=GPT5_MODEL,
        input="Explain how a hash map works.",
        reasoning={"effort": "minimal"}, max_output_tokens=4000,
        text={"verbosity": v},
    )
    print(f"{v:10s} | {r.usage.output_tokens:13d} | {len(r.output_text):6d}")

verbosity  | output_tokens |  chars
------------------------------------
low        |           379 |   1673
medium     |           657 |   3041
high       |          1411 |   6311


### Streaming: typed semantic events

Chat Completions streaming yields raw delta chunks. Responses streaming yields **typed events**
(`response.output_text.delta`, `response.completed`, and so on), so you dispatch on event type
rather than diffing chunks. Migrating a streaming UI means switching the parser.

In [14]:
stream = gateway.responses.create(
    model=GPT5_MODEL,
    input="List two real-world uses of reinforcement learning.",
    reasoning={"effort": "minimal"}, max_output_tokens=2000, stream=True,
)
event_counts, text = {}, []
for ev in stream:
    event_counts[ev.type] = event_counts.get(ev.type, 0) + 1
    if ev.type == "response.output_text.delta":
        text.append(ev.delta)

print("streamed text:\n" + "".join(text))
print("\nevent types seen:")
for t, c in sorted(event_counts.items(), key=lambda x: -x[1]):
    print(f"  {c:3d}  {t}")

streamed text:
1) Robotics: RL is used to teach robots complex motor skills—e.g., industrial robotic arms learning grasping and assembly strategies, or legged robots learning stable walking and terrain adaptation.

2) Recommendation and personalization: RL drives sequential decision-making in recommender systems and ad placement, optimizing long-term user engagement and lifetime value by learning which content to show based on user responses.

event types seen:
   76  response.output_text.delta
    2  response.output_item.added
    2  response.output_item.done
    1  response.created
    1  response.in_progress
    1  response.content_part.added
    1  response.output_text.done
    1  response.content_part.done
    1  response.completed


## Summary - gotchas and fixes

| Gotcha | Symptom | Fix |
|---|---|---|
| Sampling params | `temperature`/`top_p`/penalties -> 400 | Remove them; steer with `reasoning_effort` + `verbosity` |
| `max_tokens` | 400 unsupported parameter | Use `max_completion_tokens` / `max_output_tokens` |
| Hidden reasoning tokens | Output cost/latency jumps | Budget for them; read `reasoning_tokens`; pick the lowest workable effort |
| Empty answer | `finish_reason="length"`, blank content, still billed | Give generous output budget (reserve ~25K while tuning) |
| Effort cost/latency | Slow, expensive at high effort | Tune `reasoning_effort` per route; `none`/`minimal` for low latency |
| `system` role | (works) auto-mapped to `developer` | No change needed; do not send both roles |
| API surface | Want reasoning reuse, multi-turn, streaming events | Adopt the Responses API (`output_text`, `previous_response_id`) |

**Recommended migration path for a GPT-4o workload**

1. Decide the target: `gpt-5.1` at `reasoning_effort="none"` (or `gpt-5-mini` at `minimal`)
   for latency-sensitive paths; raise effort only where quality needs it.
2. Strip unsupported sampling params; switch `max_tokens` to `max_completion_tokens` /
   `max_output_tokens` and size it with headroom.
3. Adopt the Responses API for multi-turn and agentic flows to reuse reasoning and cut cost.
4. Rework the prompt: drop chain-of-thought prose and contradictions (they cost reasoning
   tokens); set `verbosity`. See the
   [migration research notes](../docs/gpt-4o-to-gpt-5.1-migration-research.md).

> **Cleanup (optional).** This lab leaves `gpt-4o` deployed on the core account so the gateway
> keeps serving it. To remove it:
> `az cognitiveservices account deployment delete -n aif-core-{suffix} -g rg-foundry-core-{suffix} --deployment-name gpt-4o`